# Chapter 2: Calculus & Optimization for LLMs

This notebook covers the calculus and optimization techniques that make training LLMs possible — from the chain rule that powers backpropagation to the Adam optimizer that handles the noisy, high-dimensional loss landscapes of billion-parameter models.

**Topics covered:**
1. Derivatives & the Chain Rule
2. Gradient Vector & Autograd
3. Computational Graphs
4. Backpropagation in a 2-Layer MLP
5. SGD, Momentum & Adam
6. Learning Rate Scheduling
7. Gradient Clipping
8. Adam Optimizer from Scratch

**Prerequisites:** Chapter 1 (tensors and matrix operations). PyTorch with autograd.

**Install:** `!pip install torch matplotlib` if needed.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

# Optional matplotlib (graceful fallback)
try:
    import matplotlib.pyplot as plt
    MATPLOTLIB = True
    print("matplotlib available — plots will be shown")
except ImportError:
    MATPLOTLIB = False
    print("matplotlib not available — skipping plots (install with: pip install matplotlib)")

torch.manual_seed(42)
print(f"PyTorch {torch.__version__}")

## 1. Derivatives & the Chain Rule

The **derivative** of $f(x)$ measures its instantaneous rate of change:

$$f'(x) = \frac{d}{dx}f(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

### Chain Rule

For a composition $f(g(x))$:

$$\frac{d}{dx}[f(g(x))] = f'(g(x)) \cdot g'(x)$$

This is the mathematical engine behind backpropagation — every layer is a composition of functions, and the chain rule tells us how to propagate gradients backward through each one.

### Common Activation Derivatives

| Activation | $f(x)$ | $f'(x)$ |
|---|---|---|
| Sigmoid | $\sigma(x) = \frac{1}{1+e^{-x}}$ | $\sigma(x)(1 - \sigma(x))$ |
| ReLU | $\max(0, x)$ | $\mathbf{1}[x > 0]$ |
| tanh | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $1 - \tanh^2(x)$ |
| GELU | $x \cdot \Phi(x)$ | complex, computed by autograd |

**Why sigmoid is rarely used in LLMs:** $\sigma'$ is at most $0.25$ (at $x=0$) and decays to 0 for large $|x|$. Deep networks suffer **vanishing gradients** — products of many values $< 1$ shrink exponentially. ReLU and GELU avoid this by having derivative 1 (or close) for positive inputs.

In [ ]:
# ---- Verify sigmoid derivative with autograd ----

def sigmoid_analytical_deriv(x):
    """Analytical: sigma'(x) = sigma(x) * (1 - sigma(x))"""
    s = torch.sigmoid(x)
    return s * (1 - s)

x_val = torch.tensor([[-2.0, -1.0, 0.0, 1.0, 2.0]], requires_grad=False)

# Autograd approach: compute grad of sum(sigmoid(x)) w.r.t. x
x_ag = x_val.clone().requires_grad_(True)
y = torch.sigmoid(x_ag).sum()
y.backward()
grad_autograd = x_ag.grad

# Analytical
grad_analytical = sigmoid_analytical_deriv(x_val)

print("Sigmoid derivative verification:")
print(f"  x values:    {x_val.squeeze().tolist()}")
print(f"  Autograd:    {grad_autograd.squeeze().tolist()}")
print(f"  Analytical:  {grad_analytical.squeeze().tolist()}")
print(f"  Max error:   {(grad_autograd - grad_analytical).abs().max():.2e}")

# ---- Numerical finite difference check ----
h = 1e-5
x_test = torch.tensor([0.5])
numerical_grad = (torch.sigmoid(x_test + h) - torch.sigmoid(x_test - h)) / (2 * h)
analytical_at_05 = sigmoid_analytical_deriv(x_test)
print(f"\nAt x=0.5: numerical={numerical_grad.item():.6f}, analytical={analytical_at_05.item():.6f}")

# ---- Chain rule demo: d/dx[sigmoid(3x^2 + 1)] ----
print("\n--- Chain Rule Demo: d/dx[sigmoid(3x^2 + 1)] ---")
x_cr = torch.tensor([1.0], requires_grad=True)
g_x   = 3 * x_cr**2 + 1          # inner function g(x) = 3x^2+1
f_gx  = torch.sigmoid(g_x)        # outer function f(g) = sigmoid(g)
f_gx.backward()

# Manual chain rule: df/dx = sigma'(g(x)) * g'(x) = sigma(g)*(1-sigma(g)) * 6x
x0 = 1.0
g0 = 3*x0**2 + 1     # = 4.0
gp = 6 * x0          # g'(x) = 6x
s0 = 1 / (1 + np.exp(-g0))
fp = s0 * (1 - s0) * gp
print(f"  Autograd result:     {x_cr.grad.item():.6f}")
print(f"  Manual chain rule:   {fp:.6f}")

# ---- Compare activation function derivatives ----
print("\n--- Derivative values at x in [-3, 3] ---")
x_range = torch.linspace(-3, 3, 7, requires_grad=False)
print(f"x:       {x_range.tolist()}")

# Sigmoid derivative
s = torch.sigmoid(x_range)
print(f"sigma':  {(s*(1-s)).round(decimals=3).tolist()}")

# ReLU derivative (1 for x>0, 0 otherwise)
relu_deriv = (x_range > 0).float()
print(f"ReLU':   {relu_deriv.tolist()}")

# tanh derivative
t = torch.tanh(x_range)
print(f"tanh':   {(1 - t**2).round(decimals=3).tolist()}")

## 2. Gradient Vector & Autograd

For a scalar loss $\mathcal{L}$ and parameter vector $\mathbf{w} = [w_1, \ldots, w_n]^T$, the **gradient** is:

$$\nabla_{\mathbf{w}}\mathcal{L} = \left[\frac{\partial\mathcal{L}}{\partial w_1}, \frac{\partial\mathcal{L}}{\partial w_2}, \ldots, \frac{\partial\mathcal{L}}{\partial w_n}\right]^T$$

The gradient points in the direction of **steepest ascent** of $\mathcal{L}$. Subtracting it moves us toward lower loss (gradient descent).

### PyTorch Autograd

PyTorch uses **reverse-mode automatic differentiation**:

1. **Forward pass:** compute $\mathcal{L}$ while recording operations in a computational graph
2. **Backward pass:** call `.backward()` to traverse the graph in reverse, applying the chain rule
3. **Accumulation:** gradients accumulate in `.grad` attribute of leaf tensors with `requires_grad=True`
4. **Zero grad:** call `.zero_grad()` before each backward pass to reset accumulated gradients

**Important:** Gradients **accumulate** by default (they are added to `.grad`). Always zero them before computing new gradients in a training loop, or you get incorrect updates.

In [ ]:
torch.manual_seed(42)

# ---- Simple linear regression gradient ----
# Generate data: y = 2x + 1 + noise
N = 50
X = torch.randn(N, 3)          # 50 samples, 3 features
w_true = torch.tensor([2.0, -1.0, 0.5])
y = X @ w_true + 0.1 * torch.randn(N)

# Parameters to optimize
w = torch.zeros(3, requires_grad=True)   # start at zero

# Forward pass: MSE loss
y_pred = X @ w
L = (y_pred - y).pow(2).mean()
print(f"Loss before backward: {L.item():.4f}")
print(f"w.grad before backward: {w.grad}")

# Backward pass
L.backward()
print(f"\nw.grad after backward: {w.grad}")
print(f"  (Analytical gradient: dL/dw = 2/N * X^T (Xw - y))")

# Verify analytically: dL/dw = 2/N * X^T (Xw - y)
with torch.no_grad():
    residual = X @ w - y
    grad_manual = 2 * X.T @ residual / N
print(f"Manual gradient:  {grad_manual}")
print(f"Autograd gradient: {w.grad}")
print(f"Match: {torch.allclose(w.grad, grad_manual, atol=1e-5)}")

# ---- Demonstrate gradient accumulation (common pitfall) ----
print("\n--- Gradient Accumulation Demo ---")
w2 = torch.ones(3, requires_grad=True)
for i in range(3):
    loss = (X @ w2 - y).pow(2).mean()
    loss.backward()  # gradients accumulate!
    print(f"  After backward {i+1}: grad norm = {w2.grad.norm():.4f}")

# Now with zero_grad
print("\nWith zero_grad before each backward:")
w3 = torch.ones(3, requires_grad=True)
for i in range(3):
    if w3.grad is not None:
        w3.grad.zero_()   # reset to zero
    loss = (X @ w3 - y).pow(2).mean()
    loss.backward()
    print(f"  After backward {i+1}: grad norm = {w3.grad.norm():.4f}  (same each time)")

## 3. Computational Graph

PyTorch builds a **dynamic computational graph** during the forward pass. Each operation creates a node that stores:
- The output tensor
- References to input tensors
- A gradient function (`grad_fn`) that computes the local Jacobian

### Forward-Backward for Linear Regression

Consider $L = (xw + b - y)^2$ for scalars:

**Forward pass:**
$$z_1 = xw, \quad z_2 = z_1 + b, \quad r = z_2 - y, \quad L = r^2$$

**Backward pass** (chain rule):
$$\frac{\partial L}{\partial r} = 2r$$
$$\frac{\partial L}{\partial z_2} = \frac{\partial L}{\partial r} \cdot 1 = 2r$$
$$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial z_2} \cdot 1 = 2r$$
$$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial z_2} \cdot 1 = 2r$$
$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial z_1} \cdot x = 2rx$$

The backward pass is a **single pass through the graph in reverse** — every gradient is computed once.

In [ ]:
# ---- Step-by-step manual backprop for L = (xw + b - y)^2 ----
torch.manual_seed(0)
x_val = 3.0
y_val = 7.0

# Parameters
w_scalar = torch.tensor(1.5, requires_grad=True)
b_scalar = torch.tensor(0.5, requires_grad=True)
x_s = torch.tensor(x_val)
y_s = torch.tensor(y_val)

# Forward pass (autograd tracks the graph)
z1 = x_s * w_scalar      # x * w
z2 = z1 + b_scalar       # x*w + b
r  = z2 - y_s            # residual
L  = r ** 2              # squared loss

print(f"Forward pass:")
print(f"  z1 = x*w = {z1.item():.4f}")
print(f"  z2 = z1+b = {z2.item():.4f}")
print(f"  r  = z2-y = {r.item():.4f}")
print(f"  L  = r^2  = {L.item():.4f}")

# Autograd backward
L.backward()
print(f"\nAutograd gradients:")
print(f"  dL/dw = {w_scalar.grad.item():.4f}")
print(f"  dL/db = {b_scalar.grad.item():.4f}")

# Manual backward
r_val  = r.item()
dL_dr  = 2 * r_val
dL_dz2 = dL_dr * 1
dL_db  = dL_dz2 * 1
dL_dz1 = dL_dz2 * 1
dL_dw  = dL_dz1 * x_val
print(f"\nManual gradients:")
print(f"  dL/dr  = 2r = {dL_dr:.4f}")
print(f"  dL/dz2 = {dL_dz2:.4f}")
print(f"  dL/db  = {dL_db:.4f}")
print(f"  dL/dw  = 2r*x = {dL_dw:.4f}")
print(f"\nMatch: dL/dw {torch.isclose(w_scalar.grad, torch.tensor(dL_dw))}")
print(f"Match: dL/db {torch.isclose(b_scalar.grad, torch.tensor(dL_db))}")

# Inspect grad_fn chain
print(f"\nComputational graph (grad_fn chain):")
print(f"  L.grad_fn = {L.grad_fn}")
print(f"  r.grad_fn = {r.grad_fn}")
print(f"  z2.grad_fn = {z2.grad_fn}")
print(f"  z1.grad_fn = {z1.grad_fn}")

## 4. Backpropagation in a 2-Layer MLP

A 2-layer MLP computes:

$$\mathbf{h} = \text{ReLU}(\mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)})$$
$$\hat{\mathbf{y}} = \mathbf{h}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}$$
$$\mathcal{L} = \frac{1}{N}\|\hat{\mathbf{y}} - \mathbf{y}\|^2_F$$

### Backprop Derivation

**Output layer gradients:**
$$\frac{\partial\mathcal{L}}{\partial\hat{\mathbf{y}}} = \frac{2}{N}(\hat{\mathbf{y}} - \mathbf{y})$$
$$\frac{\partial\mathcal{L}}{\partial\mathbf{W}^{(2)}} = \mathbf{h}^T \frac{\partial\mathcal{L}}{\partial\hat{\mathbf{y}}}$$

**Hidden layer gradients** (chain rule through ReLU):
$$\frac{\partial\mathcal{L}}{\partial\mathbf{h}} = \frac{\partial\mathcal{L}}{\partial\hat{\mathbf{y}}} (\mathbf{W}^{(2)})^T$$
$$\frac{\partial\mathcal{L}}{\partial\mathbf{z}} = \frac{\partial\mathcal{L}}{\partial\mathbf{h}} \odot \mathbf{1}[\mathbf{z} > 0]$$
$$\frac{\partial\mathcal{L}}{\partial\mathbf{W}^{(1)}} = \mathbf{X}^T \frac{\partial\mathcal{L}}{\partial\mathbf{z}}$$

where $\odot$ is the elementwise (Hadamard) product. The ReLU derivative $\mathbf{1}[\mathbf{z} > 0]$ is a **binary mask** — gradients flow through active neurons (those with $z > 0$) and are zeroed for inactive ones.

In [ ]:
torch.manual_seed(42)

# ---- Build a 2-layer MLP ----
N, d_in, d_hidden, d_out = 32, 8, 16, 4

model = nn.Sequential(
    nn.Linear(d_in, d_hidden),
    nn.ReLU(),
    nn.Linear(d_hidden, d_out)
)

# Random data
X_mlp = torch.randn(N, d_in)
y_mlp = torch.randn(N, d_out)

# Forward pass
y_hat = model(X_mlp)
loss = F.mse_loss(y_hat, y_mlp)
print(f"Forward pass: input {X_mlp.shape} -> output {y_hat.shape}")
print(f"Loss: {loss.item():.4f}")

# Backward pass
loss.backward()

print("\nGradient shapes and norms (after backward):")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"  {name:20s}: shape={str(param.grad.shape):15s}  grad_norm={param.grad.norm():.4f}")

# ---- Manual backprop for comparison ----
print("\n--- Manual backprop verification ---")
W1 = model[0].weight.data   # (d_hidden, d_in)
b1 = model[0].bias.data     # (d_hidden,)
W2 = model[2].weight.data   # (d_out, d_hidden)
b2 = model[2].bias.data     # (d_out,)

with torch.no_grad():
    # Forward
    z1 = X_mlp @ W1.T + b1       # (N, d_hidden)
    h  = F.relu(z1)              # (N, d_hidden)
    z2 = h @ W2.T + b2           # (N, d_out)

    # Backward
    dL_dz2 = 2 * (z2 - y_mlp) / N   # (N, d_out)
    dL_dW2 = dL_dz2.T @ h            # (d_out, d_hidden)
    dL_db2 = dL_dz2.sum(dim=0)       # (d_out,)
    dL_dh  = dL_dz2 @ W2             # (N, d_hidden)
    dL_dz1 = dL_dh * (z1 > 0).float()  # ReLU gate
    dL_dW1 = dL_dz1.T @ X_mlp        # (d_hidden, d_in)
    dL_db1 = dL_dz1.sum(dim=0)       # (d_hidden,)

    print(f"  W1 grad match: {torch.allclose(model[0].weight.grad, dL_dW1, atol=1e-4)}")
    print(f"  b1 grad match: {torch.allclose(model[0].bias.grad,   dL_db1, atol=1e-4)}")
    print(f"  W2 grad match: {torch.allclose(model[2].weight.grad, dL_dW2, atol=1e-4)}")
    print(f"  b2 grad match: {torch.allclose(model[2].bias.grad,   dL_db2, atol=1e-4)}")

## 5. SGD, Momentum & Adam

### Stochastic Gradient Descent (SGD)

The simplest optimizer. At each step:

$$\mathbf{w}_t \leftarrow \mathbf{w}_{t-1} - \eta \nabla_{\mathbf{w}}\mathcal{L}$$

**Problems:** slow convergence, oscillates in high-curvature directions, sensitive to learning rate.

### SGD with Momentum

Accumulate a velocity vector to smooth out noisy gradients:

$$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + (1-\beta) \nabla\mathcal{L}$$
$$\mathbf{w}_t = \mathbf{w}_{t-1} - \eta \mathbf{v}_t$$

Typical $\beta = 0.9$. The velocity exponentially averages recent gradients.

### Adam (Adaptive Moment Estimation)

Adam maintains **per-parameter** adaptive learning rates using first and second moment estimates:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \quad \text{(first moment / mean)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2 \quad \text{(second moment / variance)}$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \quad \text{(bias correction)}$$
$$\mathbf{w}_t = \mathbf{w}_{t-1} - \frac{\eta\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Default hyperparameters: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

### AdamW

**AdamW** decouples weight decay from the gradient update (Loshchilov & Hutter, 2019):

$$\mathbf{w}_t = \mathbf{w}_{t-1} - \eta\left(\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \mathbf{w}_{t-1}\right)$$

This is the default optimizer for training GPT, LLaMA, and most modern LLMs.

In [ ]:
torch.manual_seed(42)

# ---- Synthetic regression problem ----
N_train = 200
d_feat  = 20
X_train = torch.randn(N_train, d_feat)
w_true_reg = torch.randn(d_feat)
y_train = X_train @ w_true_reg + 0.1 * torch.randn(N_train)

def train_model(optimizer_class, lr=1e-2, n_steps=500, **opt_kwargs):
    """Train a linear model and return loss history."""
    model_w = nn.Linear(d_feat, 1, bias=True)
    nn.init.zeros_(model_w.weight)
    nn.init.zeros_(model_w.bias)
    opt = optimizer_class(model_w.parameters(), lr=lr, **opt_kwargs)
    losses = []
    for step in range(n_steps):
        opt.zero_grad()
        pred = model_w(X_train).squeeze()
        loss = F.mse_loss(pred, y_train)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step % 100 == 0 or step == n_steps - 1:
            print(f"    step {step:4d}: loss = {loss.item():.6f}")
    return losses

print("=== SGD (lr=0.05) ===")
losses_sgd = train_model(optim.SGD, lr=0.05)

print("\n=== Adam (lr=0.01) ===")
losses_adam = train_model(optim.Adam, lr=0.01)

print("\n=== AdamW (lr=0.01, weight_decay=0.01) ===")
losses_adamw = train_model(optim.AdamW, lr=0.01, weight_decay=0.01)

# Summary
print("\n--- Final loss comparison ---")
print(f"  SGD   final loss: {losses_sgd[-1]:.6f}")
print(f"  Adam  final loss: {losses_adam[-1]:.6f}")
print(f"  AdamW final loss: {losses_adamw[-1]:.6f}")

if MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(losses_sgd,   label='SGD',   alpha=0.8)
    ax.semilogy(losses_adam,  label='Adam',  alpha=0.8)
    ax.semilogy(losses_adamw, label='AdamW', alpha=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title('Optimizer Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Learning Rate Scheduling

A fixed learning rate is rarely optimal. Modern LLM training uses **learning rate schedules** that vary $\eta$ over the course of training.

### Linear Warmup

Start with a very small LR and linearly increase to $\eta_{\max}$ over $T_{\text{warm}}$ steps:

$$\eta_t = \eta_{\max} \cdot \frac{t}{T_{\text{warm}}}, \quad t < T_{\text{warm}}$$

**Why warmup?** At initialization, model weights are random. A large LR at step 0 causes wildly erratic updates. Warmup lets the optimizer accumulate meaningful moment estimates before taking large steps.

### Cosine Annealing

After warmup, decay the LR following a cosine curve:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{t\pi}{T}\right)\right)$$

The cosine shape provides fast initial decay followed by very slow decay near the end — spending more optimization budget in the "good" region near the minimum.

### Warmup + Cosine (Standard LLM Schedule)

```
LR
^     /\__________
|    /             \_
|   /                \_____
+--|--------|------------|---> step
   0   T_warm          T_total
```

This is the schedule used to train GPT-3, LLaMA, and most large language models.

In [ ]:
torch.manual_seed(42)

# A simple 2-param model (just to have an optimizer)
dummy_model = nn.Linear(4, 2)
base_lr = 1e-3
total_steps = 200
warmup_steps = 20

# ---- Cosine Annealing LR ----
print("--- CosineAnnealingLR ---")
opt_cos = optim.Adam(dummy_model.parameters(), lr=base_lr)
sched_cos = optim.lr_scheduler.CosineAnnealingLR(opt_cos, T_max=total_steps, eta_min=1e-6)

lr_cos = []
for step in range(total_steps):
    lr_cos.append(opt_cos.param_groups[0]['lr'])
    opt_cos.zero_grad()
    loss_dummy = dummy_model(torch.randn(8, 4)).mean()
    loss_dummy.backward()
    opt_cos.step()
    sched_cos.step()

print(f"  LR at step 0:   {lr_cos[0]:.2e}")
print(f"  LR at step 50:  {lr_cos[50]:.2e}")
print(f"  LR at step 100: {lr_cos[100]:.2e}")
print(f"  LR at step 199: {lr_cos[199]:.2e}")

# ---- Linear Warmup + Cosine Decay (SequentialLR) ----
print("\n--- Warmup + CosineAnnealing (SequentialLR) ---")
dummy_model2 = nn.Linear(4, 2)
opt_seq = optim.AdamW(dummy_model2.parameters(), lr=base_lr)

# LinearLR for warmup: LR goes from start_factor * base_lr -> end_factor * base_lr
sched_warmup = optim.lr_scheduler.LinearLR(
    opt_seq, start_factor=1e-3, end_factor=1.0, total_iters=warmup_steps
)
# CosineAnnealingLR for decay phase
sched_cosine = optim.lr_scheduler.CosineAnnealingLR(
    opt_seq, T_max=total_steps - warmup_steps, eta_min=1e-6
)
# Combine: first warmup_steps use warmup, then cosine
sched_combined = optim.lr_scheduler.SequentialLR(
    opt_seq,
    schedulers=[sched_warmup, sched_cosine],
    milestones=[warmup_steps]
)

lr_combined = []
for step in range(total_steps):
    lr_combined.append(opt_seq.param_groups[0]['lr'])
    opt_seq.zero_grad()
    loss_d2 = dummy_model2(torch.randn(8, 4)).mean()
    loss_d2.backward()
    opt_seq.step()
    sched_combined.step()

print(f"  LR at step 0:   {lr_combined[0]:.2e}  (warmup start)")
print(f"  LR at step 10:  {lr_combined[10]:.2e}  (mid warmup)")
print(f"  LR at step 20:  {lr_combined[20]:.2e}  (warmup end ~ base_lr)")
print(f"  LR at step 100: {lr_combined[100]:.2e}  (mid cosine decay)")
print(f"  LR at step 199: {lr_combined[199]:.2e}  (end of cosine decay)")

# Print full LR schedule at key checkpoints
print("\nLR schedule (every 20 steps):")
for step in range(0, total_steps, 20):
    bar_len = int(lr_combined[step] / base_lr * 30)
    bar = '|' * bar_len
    print(f"  step {step:3d}: {lr_combined[step]:.2e}  {bar}")

if MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(lr_cos)
    axes[0].set_title('Cosine Annealing')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Learning Rate')
    axes[1].plot(lr_combined)
    axes[1].axvline(x=warmup_steps, color='r', linestyle='--', label=f'warmup end (step {warmup_steps})')
    axes[1].set_title('Warmup + Cosine Decay')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Learning Rate')
    axes[1].legend()
    for ax in axes:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Gradient Clipping

Deep networks (especially RNNs and transformers with long sequences) can suffer from **exploding gradients** — a cascade where gradients grow exponentially as they backpropagate through many layers.

### Global Gradient Clipping

Rescale the **entire gradient vector** so its L2 norm does not exceed a threshold $\tau$:

$$\mathbf{g} \leftarrow \mathbf{g} \cdot \frac{\tau}{\max(\tau, \|\mathbf{g}\|_2)}$$

This preserves the **direction** of the gradient while limiting its magnitude.

**Clipping threshold $\tau$:** typically set to 1.0 for transformers (GPT, LLaMA, T5 all use `max_grad_norm=1.0`).

### Why Global (Not Per-Parameter)?

Clipping each parameter's gradient independently changes the relative scaling between parameters, which can distort the update direction. Global clipping treats all parameters as a single gradient vector, preserving relative magnitudes.

### Gradient Norm as a Training Signal

Logging the pre-clip gradient norm during training is a valuable diagnostic:
- Consistently large norms → instability, reduce LR or reduce batch size
- Sudden spike in gradient norm → a "bad" batch or numerical instability
- Norm decaying to near-zero → potential vanishing gradient issue

In [ ]:
torch.manual_seed(42)

# ---- Setup a simple model ----
model_clip = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Linear(32, 8)
)

# ---- Artificially inflate gradients to simulate exploding gradients ----
x_clip = torch.randn(4, 16)
y_clip = torch.randn(4, 8)
opt_clip = optim.SGD(model_clip.parameters(), lr=0.01)

opt_clip.zero_grad()
loss_clip = F.mse_loss(model_clip(x_clip), y_clip)
loss_clip.backward()

# Manually scale up gradients to simulate exploding gradients
with torch.no_grad():
    for p in model_clip.parameters():
        if p.grad is not None:
            p.grad *= 1000.0   # simulate exploding gradient

# Compute total gradient norm before clipping
total_norm_before = 0.0
for p in model_clip.parameters():
    if p.grad is not None:
        param_norm = p.grad.data.norm(2)
        total_norm_before += param_norm.item() ** 2
total_norm_before = total_norm_before ** 0.5

print(f"Gradient norm BEFORE clipping: {total_norm_before:.2f}")

# Apply gradient clipping
max_norm = 1.0
actual_norm = torch.nn.utils.clip_grad_norm_(model_clip.parameters(), max_norm=max_norm)

# Compute total gradient norm after clipping
total_norm_after = 0.0
for p in model_clip.parameters():
    if p.grad is not None:
        param_norm = p.grad.data.norm(2)
        total_norm_after += param_norm.item() ** 2
total_norm_after = total_norm_after ** 0.5

print(f"Gradient norm AFTER  clipping: {total_norm_after:.2f}  (target: {max_norm})")
print(f"Clipping factor applied: {max_norm / total_norm_before:.6f}")
print(f"clip_grad_norm_ returned: {actual_norm:.2f}  (pre-clip norm)")

# Verify direction is preserved
print(f"\nGradient direction preserved (ratio of norms = max_norm): {abs(total_norm_after - max_norm) < 1e-4}")

# ---- Demonstrate with normal gradients (no clipping needed) ----
print("\n--- Normal gradients (no explosion) ---")
model_normal = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 8))
opt_normal = optim.Adam(model_normal.parameters(), lr=0.001)
opt_normal.zero_grad()
loss_normal = F.mse_loss(model_normal(x_clip), y_clip)
loss_normal.backward()

norm_before_normal = torch.nn.utils.clip_grad_norm_(model_normal.parameters(), max_norm=1.0)
print(f"Normal gradient norm: {norm_before_normal:.4f}  (< 1.0, so no clipping applied)")

# ---- Training loop with gradient clipping ----
print("\n--- Training with gradient clipping (monitoring norm) ---")
model_loop = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 8))
opt_loop = optim.AdamW(model_loop.parameters(), lr=0.01)
X_loop = torch.randn(64, 16)
y_loop = torch.randn(64, 8)

for step in range(5):
    opt_loop.zero_grad()
    loss_loop = F.mse_loss(model_loop(X_loop), y_loop)
    loss_loop.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model_loop.parameters(), max_norm=1.0)
    opt_loop.step()
    print(f"  step {step}: loss={loss_loop.item():.4f}  grad_norm={grad_norm:.4f}")

## 8. Adam Optimizer from Scratch

Implementing Adam from scratch deepens understanding of exactly what the optimizer does at each step.

### Full Adam Algorithm

**Hyperparameters:** $\eta$ (learning rate), $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$

**Initialize:** $m_0 = 0$, $v_0 = 0$, $t = 0$

**At each step $t$:**

1. $t \leftarrow t + 1$
2. $g_t = \nabla_{\mathbf{w}}\mathcal{L}_t(\mathbf{w}_{t-1})$ — compute gradient
3. $m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$ — update biased first moment (mean)
4. $v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$ — update biased second moment (uncentered variance)
5. $\hat{m}_t = m_t / (1 - \beta_1^t)$ — bias-corrected first moment
6. $\hat{v}_t = v_t / (1 - \beta_2^t)$ — bias-corrected second moment
7. $\mathbf{w}_t = \mathbf{w}_{t-1} - \eta \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$

### Why Bias Correction?

At $t=1$: $m_1 = (1-\beta_1)g_1$. If $\beta_1 = 0.9$, then $m_1 = 0.1 g_1$ — a factor of 10 smaller than the true gradient. Bias correction ($\hat{m}_1 = m_1 / 0.1 = g_1$) compensates for this initialization bias. The correction becomes negligible for large $t$ since $\beta^t \to 0$.

In [ ]:
class AdamOptimizer:
    """
    Adam optimizer implemented from scratch.
    Matches torch.optim.Adam behavior exactly.
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay

        # State: one m and v per parameter
        self.m = [torch.zeros_like(p) for p in self.params]   # first moment
        self.v = [torch.zeros_like(p) for p in self.params]   # second moment
        self.t = 0   # step counter

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        self.t += 1
        # Bias correction factors
        bc1 = 1 - self.beta1 ** self.t
        bc2 = 1 - self.beta2 ** self.t

        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                g = p.grad

                # Optional weight decay (L2 regularization added to gradient)
                if self.weight_decay != 0:
                    g = g + self.weight_decay * p

                # Update biased first and second moments
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g * g

                # Bias-corrected estimates
                m_hat = self.m[i] / bc1
                v_hat = self.v[i] / bc2

                # Parameter update
                p -= self.lr * m_hat / (v_hat.sqrt() + self.eps)


# ---- Verify our Adam matches torch.optim.Adam ----
print("=== Verifying custom Adam against torch.optim.Adam ===")
torch.manual_seed(42)

N_v, d_v = 100, 10
X_v = torch.randn(N_v, d_v)
y_v = torch.randn(N_v)

# Model A: custom Adam
torch.manual_seed(0)
model_a = nn.Linear(d_v, 1, bias=True)
opt_custom = AdamOptimizer(model_a.parameters(), lr=1e-3)

# Model B: torch.optim.Adam with identical init
torch.manual_seed(0)
model_b = nn.Linear(d_v, 1, bias=True)
opt_torch_adam = optim.Adam(model_b.parameters(), lr=1e-3)

n_steps = 50
losses_custom = []
losses_torch  = []

for step in range(n_steps):
    # Custom Adam step
    opt_custom.zero_grad()
    loss_a = F.mse_loss(model_a(X_v).squeeze(), y_v)
    loss_a.backward()
    opt_custom.step()

    # PyTorch Adam step
    opt_torch_adam.zero_grad()
    loss_b = F.mse_loss(model_b(X_v).squeeze(), y_v)
    loss_b.backward()
    opt_torch_adam.step()

    losses_custom.append(loss_a.item())
    losses_torch.append(loss_b.item())

    if step % 10 == 0 or step == n_steps - 1:
        diff = abs(loss_a.item() - loss_b.item())
        print(f"  step {step:3d}: custom={loss_a.item():.6f}  torch={loss_b.item():.6f}  diff={diff:.2e}")

# Final weight comparison
print("\nFinal weight comparison:")
w_diff = (model_a.weight.data - model_b.weight.data).abs().max().item()
b_diff = (model_a.bias.data   - model_b.bias.data).abs().max().item()
print(f"  Max weight difference: {w_diff:.2e}")
print(f"  Max bias difference:   {b_diff:.2e}")
print(f"  Weights match (atol=1e-5): {torch.allclose(model_a.weight.data, model_b.weight.data, atol=1e-5)}")

# ---- Demonstrate bias correction effect ----
print("\n--- Bias Correction Demo ---")
beta1, beta2 = 0.9, 0.999
print(f"Step  | 1-beta1^t (bc1) | 1-beta2^t (bc2) | Effective LR scaling")
print("-" * 65)
for t_demo in [1, 2, 5, 10, 50, 100, 1000]:
    bc1 = 1 - beta1**t_demo
    bc2 = 1 - beta2**t_demo
    # The effective LR is scaled by sqrt(bc2)/bc1 relative to nominal lr
    eff_scale = (bc2**0.5) / bc1
    print(f"  {t_demo:4d}  |   {bc1:.6f}    |   {bc2:.6f}    | {eff_scale:.4f}")
print("(At t=1, bias correction amplifies LR significantly; converges to 1.0 for large t)")

## Summary

This chapter covered the calculus and optimization tools that enable training LLMs:

| Concept | LLM Application |
|---|---|
| Chain rule | Backpropagation through layers |
| Autograd | Automatic gradient computation |
| Computational graphs | Dynamic differentiation in PyTorch |
| Backprop in MLP | Foundation for transformer training |
| Adam optimizer | Default for GPT, LLaMA, most LLMs |
| AdamW | Decoupled weight decay, standard for LLMs |
| Warmup + cosine LR | LLM training schedule |
| Gradient clipping | Prevents training instability |

**Key takeaways:**

1. **Backprop is just the chain rule** — PyTorch automates it via reverse-mode AD
2. **Adam adapts per-parameter learning rates** — parameters with high gradient variance get smaller updates
3. **Bias correction matters early** in training — without it, Adam underestimates true gradients in the first few steps
4. **Warmup is essential** for stable large-batch training — prevents destructive early updates
5. **Gradient clipping to norm 1.0** is standard practice for transformer training

**Next:** Chapter 3 covers Probability & Statistics — the mathematical framework for language modeling, entropy, cross-entropy loss, and sampling strategies (temperature, top-k, nucleus sampling).